In [1]:
"""
Simple Pharmacophore Modeling Example using RDKit
This example demonstrates:
1. Building a feature factory for pharmacophore features
2. Extracting features from multiple ligand molecules
3. Constructing a basic pharmacophore model by finding common features
4. Applying the model to screen a query molecule

Features include: H-bond donors/acceptors, aromatics, hydrophobics, etc.
Note: This is a 2D/3D hybrid approach. For full 3D pharmacophore, consider distances between features.
"""

from rdkit import Chem
from rdkit.Chem import ChemicalFeatures, AllChem, Draw
from rdkit import RDConfig
import os


# =============================================================================
# 1. Prepare feature factory
# =============================================================================

# Load default feature definition file (comes with RDKit)
fdef_path = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
factory = ChemicalFeatures.BuildFeatureFactory(fdef_path)

print("Feature factory loaded with definitions for donors, acceptors, aromatics, etc.")


Feature factory loaded with definitions for donors, acceptors, aromatics, etc.


In [2]:
# =============================================================================
# 2. Define known active ligands (HER2 inhibitors)
# =============================================================================

# SMILES for known HER2 inhibitors
active_smiles = [
    'CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)OCC5=CC(=CC=C5)F)Cl',  # Lapatinib
    'CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=CC=CC=N4)Cl)C#N)NC(=O)C=CCN(C)C',  # Neratinib
    'CC1=C(C=CC(=C1)NC2=NC=NC3=C2C=C(C=C3)NC4=NC(CO4)(C)C)OC5=CC6=NC=NN6C=C5',  # Tucatinib
]

# Convert to RDKit molecules and generate 3D conformers
actives = []
for smi in active_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mol = Chem.AddHs(mol)  # Add hydrogens for accurate features
        AllChem.EmbedMolecule(mol, randomSeed=42)  # Generate 3D conformation
        AllChem.MMFFOptimizeMolecule(mol)  # Optimize
        actives.append(mol)

print(f"Processed {len(actives)} active ligands with 3D conformations.")


Processed 3 active ligands with 3D conformations.


In [3]:
# =============================================================================
# 3. Extract pharmacophore features from actives
# =============================================================================

# Function to get feature types and positions
def get_features(mol, factory):
    feats = factory.GetFeaturesForMol(mol)
    feature_info = []
    for feat in feats:
        pos = feat.GetPos()  # 3D position
        feature_info.append({
            'type': feat.GetFamily(),
            'pos': (pos.x, pos.y, pos.z)
        })
    return feature_info

# Extract features for all actives
all_features = [get_features(mol, factory) for mol in actives]

# Display extracted features for first molecule
print("\nFeatures for first active (Lapatinib):")
for feat in all_features[0]:
    print(f"- {feat['type']} at position {feat['pos']}")



Features for first active (Lapatinib):
- Donor at position (7.737068461438972, 2.1567175300733785, 1.8726860839084727)
- Donor at position (-0.897882819263044, -0.5408074320899958, -0.5864595086478811)
- Acceptor at position (5.788136459451726, -2.430859904494464, 2.8839570996086152)
- Acceptor at position (8.071965313068644, -2.254766073396878, 1.8372308215450297)
- Acceptor at position (1.257678040179081, -1.4787800087392502, -4.013805430441197)
- Acceptor at position (-0.7600019982420039, -1.5466379415644622, -2.747117988683757)
- Acceptor at position (-6.402900343890992, -0.8742363995621073, 0.45597293663750404)
- Acceptor at position (-11.973852040413535, 1.2008236826428105, -0.1804859161421223)
- PosIonizable at position (7.737068461438972, 2.1567175300733785, 1.8726860839084727)
- Aromatic at position (4.4970914334642655, 2.337047844327702, 0.5738002455278065)
- Aromatic at position (2.4831160709266182, 0.05595011870175379, -2.0834772081163537)
- Aromatic at position (0.5493091

In [4]:
# =============================================================================
# 4. Construct simple pharmacophore model (common features)
# =============================================================================

# Find common feature types across all actives (basic model: set of required features)
from collections import Counter

# Collect all feature types
feature_types = [feat['type'] for features in all_features for feat in features]

# Find features present in all actives (intersection)
common_types = set(feature_types)
for features in all_features:
    types_in_mol = {feat['type'] for feat in features}
    common_types &= types_in_mol

print("\nCommon pharmacophore features across all actives:")
for ftype in common_types:
    print(f"- {ftype}")

# For a more advanced model, we could compute average positions or use rdkit.Chem.Pharm3D
# Here, the model is defined as requiring at least these feature types



Common pharmacophore features across all actives:
- Aromatic
- Hydrophobe
- Acceptor
- Donor
- LumpedHydrophobe


In [5]:
# =============================================================================
# 5. Application: Screen a query molecule against the model
# =============================================================================

# Query molecule (e.g., a potential hit or decoy)
query_smiles = 'CC(=O)OC1=CC=CC=C1C(=O)O'  # Aspirin (unlikely to match HER2 pharmacophore)
query_mol = Chem.MolFromSmiles(query_smiles)
query_mol = Chem.AddHs(query_mol)
AllChem.EmbedMolecule(query_mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(query_mol)

# Extract query features
query_features = get_features(query_mol, factory)

# Check if query has all common features (simple matching)
query_types = {feat['type'] for feat in query_features}
matches = all(ftype in query_types for ftype in common_types)

print("\nQuery molecule screening result:")
print(f"Query SMILES: {query_smiles}")
print(f"Features in query: {query_types}")
print(f"Matches pharmacophore model: {matches}")

if matches:
    print("Potential hit: Proceed to further validation.")
else:
    print("Does not match: Unlikely to be active.")

# Optional: Visualize molecules
# Draw.MolsToGridImage(actives + [query_mol], legends=['Lapatinib', 'Neratinib', 'Tucatinib', 'Query'])

# =============================================================================
# Notes:
# - This is a basic example focusing on feature types. Real pharmacophore models include distances/tolerances between features.
# - For advanced 3D pharmacophore: Use rdkit.Chem.Pharm3D to build EmbedPharmacophore from multiple conformers.
# - Applications: Use the model for virtual screening by checking if database molecules match the features and spatial arrangements.
# - Extend: Compute feature distances and create a pharmacophore query for alignment/searching.



Query molecule screening result:
Query SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
Features in query: {'Aromatic', 'Hydrophobe', 'NegIonizable', 'Acceptor', 'Donor', 'LumpedHydrophobe'}
Matches pharmacophore model: True
Potential hit: Proceed to further validation.
